In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Kaggle 必备工具
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 加载数据 (请确保路径正确)
df = pd.read_csv('your_path.cleaned_superstore.csv', encoding='windows-1252')

In [3]:
# 1. 加载数据
# 注意：根据你的 dtypes，部分数值列现在是 str，我们需要先转换
numeric_cols_to_fix = ['Sales', 'Quantity', 'Discount']
for col in numeric_cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [4]:
# 2. 特征工程与 180 天留存标签定义 (180-Day Retention Window)
# A. 确保日期是 datetime 格式 (转换一下，方便算天数)
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# B. 获取每个客户的所有唯一订单日期，并按顺序排名
# 注意：我们要找的是“第二次下单的时间”
order_ranks = df[['Customer ID', 'Order ID', 'Order_Date']].drop_duplicates()
order_ranks['rank'] = order_ranks.sort_values(['Customer ID', 'Order_Date']).groupby('Customer ID').cumcount() + 1

# C. 提取首单日期 (rank 1) 和 次单日期 (rank 2)
first_orders_time = order_ranks[order_ranks['rank'] == 1][['Customer ID', 'Order_Date']].rename(columns={'Order_Date': 'First_Date'})
second_orders_time = order_ranks[order_ranks['rank'] == 2][['Customer ID', 'Order_Date']].rename(columns={'Order_Date': 'Second_Date'})

# D. 合并并计算时间差 (Gap)
retention_logic = pd.merge(first_orders_time, second_orders_time, on='Customer ID', how='left')
retention_logic['gap'] = (retention_logic['Second_Date'] - retention_logic['First_Date']).dt.days

# E. 定义新的 Y：如果在 180 天内回来了，就是 1，否则（包括没回来的）就是 0
retention_logic['is_retained'] = ((retention_logic['gap'] > 0) & (retention_logic['gap'] <= 180)).astype(int)

# F. 获取首单的特征 (从“提取单行”改为“聚合整单”)
# a. 找到每个客户的首单日期
first_order_dates = df.groupby('Customer ID')['Order_Date'].min().reset_index()

# b. 将原始 df 与首单日期合并，筛选出首单日期的所有产品行
# 注意：此时 df_first_orders 包含了首单里的所有商品（可能是一个客户多行）
df_first_orders_all_items = pd.merge(df, first_order_dates, on=['Customer ID', 'Order_Date'])

# c. 【核心修改】按 Customer ID 分组聚合，把多行变一行，计算整单总量
df_first_orders = df_first_orders_all_items.groupby('Customer ID').agg({
    'Sales': 'sum',           # 首单总销售额
    'Quantity': 'sum',        # 首单总商品件数
    'Discount': 'mean',       # 首单平均折扣力度
    'Profit': 'sum',          # 首单总利润（为了算 Margin）
    'Region': 'first',        # 地理区域（同一单通常一致，取第一个）
    'Segment': 'first',       # 客户分段（同一客户一致，取第一个）
    'Ship Mode': 'first',     # 配送模式（同一单通常一致，取第一个）
    'Sub-Category': 'first'   # 子类别（首单可能有多个，取第一个作为代表，或选销量最大的）
}).reset_index()

# d. 【数学修正】重新计算整单的 Profit_Margin，而不是对每一行的 Margin 求平均
# 这样能更准确地反映这一单是“大赚”还是“大亏”
df_first_orders['Profit_Margin'] = df_first_orders['Profit'] / df_first_orders['Sales']
df_first_orders.replace([np.inf, -np.inf], np.nan, inplace=True)
df_first_orders['Profit_Margin'] = df_first_orders['Profit_Margin'].fillna(0)

# G. 最终合并特征与新的标签
final_df = pd.merge(
    df_first_orders, 
    retention_logic[['Customer ID', 'is_retained']], 
    on='Customer ID',
    how='inner'
)

# 打印一下结果看看，确保每个客户只有一行，且 Sales 是总和
print(f"最终特征集行数: {final_df.shape[0]} (应等于唯一客户数)")
print(final_df[['Customer ID', 'Sales', 'Profit_Margin', 'is_retained']].head())

最终特征集行数: 793 (应等于唯一客户数)
  Customer ID     Sales  Profit_Margin  is_retained
0    AA-10315   726.548       0.368073            1
1    AA-10375    16.520       0.337500            0
2    AA-10480    27.460       0.360000            0
3    AA-10645  1106.770      -0.018103            1
4    AB-10015    12.624      -0.200000            1


In [5]:
# 3. 变量选择
# 按照咱们讨论的清单，剔除 State 和不相关的 ID
numeric_features = ['Sales', 'Quantity', 'Discount', 'Profit_Margin']
categorical_features = ['Region', 'Sub-Category', 'Ship Mode', 'Segment']

X = final_df[numeric_features + categorical_features]
y = final_df['is_retained']

# 按照 Kaggle 标准：80% 学习，20% 考试
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=42, stratify=y
)

In [6]:
# 4. 构建预处理 Pipeline (The Kaggle Way)

# 数值型处理：填补中位数 + 标准化
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 类别型处理：填补缺失 + One-Hot 编码
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 组合预处理逻辑
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

# 计算不平衡比例 (留给 XGBoost 用)
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# 1. 定义我们要测试的候选模型和参数范围
n_estimators_list = [50, 100, 200, 500, 1000] # 你可以少跑一点先测试
results = []

# --- 选秀环节 A: 基础模型 Decision Tree ---
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # 加了 class_weight='balanced'，让决策树更关注少数的留存客户
    ('model', DecisionTreeClassifier(class_weight='balanced', random_state=42)) 
])
# 【关键修改】scoring='f1'
dt_score = cross_val_score(dt_pipeline, X_train, y_train, cv=5, scoring='f1').mean()
results.append({'model_name': 'Decision Tree', 'n_estimators': 'N/A', 'score': dt_score})

# --- 选秀环节 B: Random Forest 循环寻优 ---
for n in n_estimators_list:
    rf_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        # 加了 class_weight='balanced_subsample'
        ('model', RandomForestClassifier(n_estimators=n, class_weight='balanced_subsample', random_state=42))
    ])
    # 【关键修改】scoring='f1'
    score = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='f1').mean()
    results.append({'model_name': 'Random Forest', 'n_estimators': n, 'score': score})
    print(f"喵！RF (n={n}) 跑分中... F1得分: {score:.4f}")

# --- 选秀环节 C: XGBoost 循环寻优 ---
for n in n_estimators_list:
    xgb_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        # 加了 scale_pos_weight=ratio
        ('model', XGBClassifier(n_estimators=n, learning_rate=0.05, scale_pos_weight=ratio, random_state=42, eval_metric='logloss'))
    ])
    # 【关键修改】scoring='f1'
    score = cross_val_score(xgb_pipeline, X_train, y_train, cv=5, scoring='f1').mean()
    results.append({'model_name': 'XGBoost', 'n_estimators': n, 'score': score})
    print(f"喵！XGBoost (n={n}) 跑分中... F1得分: {score:.4f}")

# 2. 将所有结果转换成 DataFrame，方便一眼看穿真相
results_df = pd.DataFrame(results)
print("\n--- 全场模型选秀排行榜 (根据 F1 Score 洗牌后) ---")
print(results_df.sort_values(by='score', ascending=False))

# 3. 自动抓取最强的那个配置
best_row = results_df.loc[results_df['score'].idxmax()]
print(f"\n浴火重生的最终 MVP 是: {best_row['model_name']} (n={best_row['n_estimators']})")

喵！RF (n=50) 跑分中... F1得分: 0.3721
喵！RF (n=100) 跑分中... F1得分: 0.4083
喵！RF (n=200) 跑分中... F1得分: 0.4002
喵！RF (n=500) 跑分中... F1得分: 0.4105
喵！RF (n=1000) 跑分中... F1得分: 0.4159
喵！XGBoost (n=50) 跑分中... F1得分: 0.4562
喵！XGBoost (n=100) 跑分中... F1得分: 0.4371
喵！XGBoost (n=200) 跑分中... F1得分: 0.4252
喵！XGBoost (n=500) 跑分中... F1得分: 0.4044
喵！XGBoost (n=1000) 跑分中... F1得分: 0.4189

--- 全场模型选秀排行榜 (根据 F1 Score 洗牌后) ---
       model_name n_estimators     score
6         XGBoost           50  0.456217
0   Decision Tree          N/A  0.453435
7         XGBoost          100  0.437118
8         XGBoost          200  0.425203
10        XGBoost         1000  0.418919
5   Random Forest         1000  0.415892
4   Random Forest          500  0.410536
2   Random Forest          100  0.408322
9         XGBoost          500  0.404407
3   Random Forest          200  0.400165
1   Random Forest           50  0.372137

浴火重生的最终 MVP 是: XGBoost (n=50)
